In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error


In [12]:
df = pd.read_csv('car_fuel_efficiency_2026.csv')
df = df.fillna(0)
df.isnull().sum().sum()


np.int64(0)

In [13]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)


In [14]:
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']



In [16]:
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))
X_test = dv.transform(df_test.to_dict(orient='records'))



In [18]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

print(export_text(dt, feature_names=list(dv.get_feature_names_out())))


|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [20]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
print(round(rmse, 4))


1.8371


In [21]:
scores = []
for n in [10,50,100,150]: 
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    scores.append((n, round(rmse, 4)))
    print(f"n_estimators: {n}, RMSE: {round(rmse, 3)}")

n_estimators: 10, RMSE: 1.837
n_estimators: 50, RMSE: 1.772
n_estimators: 100, RMSE: 1.768
n_estimators: 150, RMSE: 1.769


In [23]:
scores2 = []

for d in [10, 15, 20, 25]:
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        scores2.append((d, n, rmse))

df_scores2 = pd.DataFrame(scores2, columns=['max_depth', 'n_estimators', 'rmse'])

df_scores2_pivot = df_scores2.pivot(index='max_depth', columns='n_estimators', values='rmse')
df_scores2_pivot.round(3)

n_estimators,10,50,100,150
max_depth,,,,
10,1.784,1.749,1.745,1.745
15,1.830,1.771,1.768,1.767
20,1.841,1.773,1.769,1.770
25,1.837,1.772,1.768,1.768


In [24]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

df_importance = pd.DataFrame({
    'feature': dv.get_feature_names_out(),
    'importance': rf.feature_importances_
})

df_importance.sort_values('importance', ascending=False)


,feature,importance
9,model_year,0.296001
15,vehicle_weight,0.203879
6,fuel_type=Gasoline,0.160712
8,horsepower,0.078017
4,engine_displacement,0.061005
0,acceleration,0.055201
14,origin=USA,0.042144
2,drivetrain=Front-wheel drive,0.020145
11,num_doors,0.019415
5,fuel_type=Diesel,0.017038


In [27]:
import xgboost as xgb

features = list(dv.get_feature_names_out())

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

watchlist = [(dtrain, 'train'), (dval, 'val')]


def train_xgb(eta: float) -> float:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,

        'objective': 'reg:squarederror',
        'nthread': 8,

        'seed': 1,
        'verbosity': 1,
    }

    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=100,
        evals=watchlist,
        verbose_eval=10,
    )

    y_pred = model.predict(dval)
    return root_mean_squared_error(y_val, y_pred)


for eta in [0.3, 0.1]:
    print(f'--- eta={eta} ---')
    rmse = train_xgb(eta)
    print(f'eta={eta}: val RMSE = {rmse:.4f}\n')

--- eta=0.3 ---
[0]	train-rmse:2.40653	val-rmse:2.46782
[10]	train-rmse:1.47125	val-rmse:1.73205
[20]	train-rmse:1.39368	val-rmse:1.73766
[30]	train-rmse:1.33243	val-rmse:1.74548
[40]	train-rmse:1.25326	val-rmse:1.76296
[50]	train-rmse:1.18645	val-rmse:1.77517
[60]	train-rmse:1.10265	val-rmse:1.79401
[70]	train-rmse:1.04604	val-rmse:1.80444
[80]	train-rmse:0.99487	val-rmse:1.81308
[90]	train-rmse:0.94986	val-rmse:1.82135
[99]	train-rmse:0.90106	val-rmse:1.82646
eta=0.3: val RMSE = 1.8265

--- eta=0.1 ---
[0]	train-rmse:2.74739	val-rmse:2.78635
[10]	train-rmse:1.80592	val-rmse:1.93152
[20]	train-rmse:1.55847	val-rmse:1.75076
[30]	train-rmse:1.47544	val-rmse:1.71667
[40]	train-rmse:1.43074	val-rmse:1.71289
[50]	train-rmse:1.40368	val-rmse:1.71358
[60]	train-rmse:1.37884	val-rmse:1.71688
[70]	train-rmse:1.36283	val-rmse:1.71919
[80]	train-rmse:1.34578	val-rmse:1.72100
[90]	train-rmse:1.32546	val-rmse:1.72303
[99]	train-rmse:1.31192	val-rmse:1.72481
eta=0.1: val RMSE = 1.7248

